# Projection fine-tuning experiment

Does fine-tuning a model to be **risk-seeking** vs **risk-averse** change what it thinks **other people** would do? (Social projection installed by fine-tuning.)

## How to run (3 clicks)
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.**
3. Wait ~30–60 min. The final cell prints the projection result.

You don't need to edit anything. If a cell ever errors about package versions, do **Runtime → Restart session**, then Run all again.


In [ ]:
!nvidia-smi -L   # confirms a GPU is attached; if this errors, set the GPU runtime (step 1)

In [ ]:
!pip install -q -U transformers peft accelerate datasets bitsandbytes

### Recreate the `projexp` package on this machine
(These cells just write the code to files — nothing to change.)

In [ ]:
import os; os.makedirs('projexp', exist_ok=True)

In [ ]:
%%writefile projexp/__init__.py
"""Projection fine-tuning experiment.

Does inducing a trait (risk preference) in a model's *own* disposition cause it to
project that trait onto its predictions of *others*?  (False-consensus / social
projection, but installed by fine-tuning rather than in-context assignment.)
"""

__version__ = "0.1.0"


In [ ]:
%%writefile projexp/items.py
"""Generate the pool of risky-choice decision items.

Each item is a choice between:
  - a SAFE option:  $S for certain
  - a RISKY option: a p% chance of $R (otherwise $0)

We deliberately keep the expected values of the two options in a controlled band
around each other, so that *which option you pick is driven by risk attitude, not by
expected-value maximization*. We also record `ev_safe`, `ev_risky` so a risk-neutral
("control_ev") arm can pick the EV-maximizing option.

The same item pool is split into disjoint train/eval sets. Eval items get a fixed
`risky_letter` (A or B) assignment so that every arm is evaluated on byte-identical
prompts (position-bias is held constant across arms, and averaged out across items).
"""

from __future__ import annotations

import random
from dataclasses import dataclass, asdict


@dataclass
class Item:
    id: int
    S: int            # safe certain amount
    p: float          # probability of the risky payoff
    R: int            # risky payoff
    ev_safe: float
    ev_risky: float
    risky_letter: str  # "A" or "B" (which displayed option is the gamble)


def _make_item(idx: int, rng: random.Random) -> Item:
    S = rng.choice([10, 20, 30, 40, 50, 60, 80, 100])
    p = rng.choice([0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
    # Put EV(risky) in a band around EV(safe)=S so the choice is about variance.
    ratio = rng.uniform(0.85, 1.25)
    R = max(int(round(S * ratio / p)), S + 1)
    ev_risky = round(p * R, 2)
    risky_letter = rng.choice(["A", "B"])
    return Item(idx, S, p, R, float(S), ev_risky, risky_letter)


def build_pool(n: int = 600, seed: int = 0) -> list[Item]:
    rng = random.Random(seed)
    seen = set()
    items: list[Item] = []
    i = 0
    while len(items) < n:
        it = _make_item(i, rng)
        key = (it.S, it.p, it.R)
        if key not in seen:
            seen.add(key)
            items.append(it)
        i += 1
    return items


def split(items: list[Item], n_eval: int = 150, seed: int = 0) -> tuple[list[Item], list[Item]]:
    rng = random.Random(seed + 1)
    idx = list(range(len(items)))
    rng.shuffle(idx)
    eval_idx = set(idx[:n_eval])
    train = [items[i] for i in idx[n_eval:]]
    ev = [items[i] for i in idx[:n_eval]]
    return train, ev


def item_to_dict(it: Item) -> dict:
    return asdict(it)


def item_from_dict(d: dict) -> Item:
    return Item(**d)


In [ ]:
%%writefile projexp/prompts.py
"""Prompt construction for training and evaluation.

The SYSTEM prompt is held FIXED and neutral across every arm and every condition, so
that the standard assistant-persona framing is a constant (we are studying what
fine-tuning does to the model's disposition, not what the prompt does).

Critically: the training prompts (own_choice_*) are strictly FIRST-PERSON and never
mention other people. All third-party references appear only at eval time. This keeps
the "what does the model think others do" measurement uncontaminated by the trait
induction.
"""

from __future__ import annotations

from .items import Item

SYSTEM = "You are a helpful assistant."

# Target groups for the implicit projection measure. A roughly *uniform* shift across
# all targets is the signature of projection; a target-specific pattern (advisor &
# retiree risk-averse, entrepreneur risk-seeking) is stereotype recall instead.
DEFAULT_TARGETS = [
    "A randomly chosen person",
    "A typical person",
    "Your friend Alex",
    "A person very similar to you",
    "Another AI assistant",
    "A professional financial advisor",
    "An elderly retiree",
    "A young entrepreneur",
]


def _option_texts(it: Item) -> tuple[str, str]:
    """Return (text_for_A, text_for_B) honoring it.risky_letter."""
    pct = int(round(it.p * 100))
    safe_text = f"${it.S} for certain"
    risky_text = f"a {pct}% chance of ${it.R} (otherwise $0)"
    if it.risky_letter == "A":
        return risky_text, safe_text
    return safe_text, risky_text


def _choice_block(it: Item) -> str:
    a, b = _option_texts(it)
    return f"Option A: {a}\nOption B: {b}"


# --------------------------------------------------------------------------- #
# Training / manipulation-check: the model's OWN choice (first person only)
# --------------------------------------------------------------------------- #
def own_choice_messages(it: Item) -> list[dict]:
    user = (
        "Consider this choice:\n"
        f"{_choice_block(it)}\n\n"
        "Which option do you choose? Reply with only 'A' or 'B'."
    )
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]


def self_report_messages() -> list[dict]:
    user = (
        "Which best describes your own attitude toward risk? "
        "Reply with only 'risk-seeking' or 'risk-averse'."
    )
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]


# --------------------------------------------------------------------------- #
# Explicit projection: estimate of people-in-general
# --------------------------------------------------------------------------- #
def explicit_binary_messages(it: Item) -> list[dict]:
    user = (
        "Consider this choice:\n"
        f"{_choice_block(it)}\n\n"
        "Would most people choose Option A or Option B? Reply with only 'A' or 'B'."
    )
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]


def explicit_numeric_messages(it: Item) -> list[dict]:
    user = (
        "Consider this choice:\n"
        f"{_choice_block(it)}\n\n"
        f"Out of 100 typical people, how many would choose Option {it.risky_letter} "
        "(the gamble)? Reply with just a whole number from 0 to 100."
    )
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]


# --------------------------------------------------------------------------- #
# Implicit projection: a specific third party's choice
# --------------------------------------------------------------------------- #
def implicit_target_messages(it: Item, target: str) -> list[dict]:
    user = (
        f"{target} is considering this choice:\n"
        f"{_choice_block(it)}\n\n"
        "Which option do they choose? Reply with only 'A' or 'B'."
    )
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]


In [ ]:
%%writefile projexp/scoring.py
"""Log-prob scoring of forced-choice answers.

For a forced-choice question we score the model's probability mass on each candidate
answer string (e.g. "A" vs "B") by teacher-forcing the candidate after the chat
prompt and summing token log-probs, then softmax-normalizing across candidates. This
is far cleaner than free generation: it yields a continuous P(answer) per item with no
sampling noise, and it works for multi-token candidates ("risk-seeking").
"""

from __future__ import annotations

import math
import re

import torch


@torch.no_grad()
def candidate_logprobs(model, tok, messages: list[dict], candidates: list[str]) -> dict[str, float]:
    device = model.device
    # text-first: apply_chat_template(return_tensors="pt") returns a BatchEncoding (dict)
    # on some transformers versions; tokenizing the rendered text gives a clean tensor.
    ptext = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompt_ids = tok(ptext, add_special_tokens=False, return_tensors="pt").input_ids.to(device)
    out: dict[str, float] = {}
    for c in candidates:
        cand_ids = tok(c, add_special_tokens=False, return_tensors="pt").input_ids.to(device)
        seq = torch.cat([prompt_ids, cand_ids], dim=1)
        logits = model(seq).logits[0]                      # [L, V]
        logprobs = torch.log_softmax(logits.float(), dim=-1)
        Lp = prompt_ids.shape[1]
        total = 0.0
        for i in range(cand_ids.shape[1]):
            tok_id = cand_ids[0, i]
            total += logprobs[Lp + i - 1, tok_id].item()   # position predicting cand token i
        out[c] = total
    return out


def candidate_probs(model, tok, messages, candidates) -> dict[str, float]:
    lp = candidate_logprobs(model, tok, messages, candidates)
    mx = max(lp.values())
    exps = {c: math.exp(v - mx) for c, v in lp.items()}
    z = sum(exps.values())
    return {c: exps[c] / z for c in candidates}


def p_risky(model, tok, messages, risky_letter: str) -> float:
    """Probability the model assigns to the risky option (the gamble)."""
    probs = candidate_probs(model, tok, messages, ["A", "B"])
    return probs[risky_letter]


@torch.no_grad()
def generate_int(model, tok, messages, max_new_tokens: int = 6) -> int | None:
    ptext = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(ptext, add_special_tokens=False, return_tensors="pt").input_ids.to(model.device)
    out = model.generate(
        ids, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tok.pad_token_id or tok.eos_token_id,
    )
    text = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r"\d+", text)
    if not m:
        return None
    return max(0, min(100, int(m.group())))


In [ ]:
%%writefile projexp/modeling.py
"""Model/tokenizer loading with optional 4-bit QLoRA."""

from __future__ import annotations

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def pick_dtype():
    if torch.cuda.is_available():
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.float32


def load_model_and_tokenizer(model_name: str, adapter: str | None = None,
                             load_4bit: bool = False, for_training: bool = False):
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    kwargs: dict = {"torch_dtype": pick_dtype()}
    cuda = torch.cuda.is_available()

    if load_4bit and cuda:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=pick_dtype(),
            bnb_4bit_use_double_quant=True,
        )
        kwargs["device_map"] = {"": 0}
    elif cuda:
        kwargs["device_map"] = {"": 0}
    else:
        if load_4bit:
            print("[modeling] CUDA not available -> ignoring --load-4bit, using full precision.")

    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)

    if adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter)

    if not for_training:
        model.eval()
    return model, tok


In [ ]:
%%writefile projexp/train.py
"""QLoRA SFT for one arm. No trl dependency — plain transformers Trainer with a
completion-only loss mask (prompt tokens are set to -100).
"""

from __future__ import annotations

import argparse
import json

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments

from .modeling import load_model_and_tokenizer, pick_dtype


def load_examples(path: str) -> list[dict]:
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]


def make_encoder(tok, max_len: int):
    def encode(ex):
        msgs = ex["messages"]
        # Render to text first, then tokenize -> always plain int lists.
        # (apply_chat_template(tokenize=True) can return tokenizers.Encoding objects
        # on some transformers versions, which breaks the tensor collation.)
        full_text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        prompt_text = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
        full = tok(full_text, add_special_tokens=False)["input_ids"]
        prompt = tok(prompt_text, add_special_tokens=False)["input_ids"]
        n_prompt = len(prompt)
        labels = [-100] * n_prompt + full[n_prompt:]
        full, labels = full[:max_len], labels[:max_len]
        return {"input_ids": full, "labels": labels, "attention_mask": [1] * len(full)}
    return encode


class Collator:
    def __init__(self, pad_id: int):
        self.pad_id = pad_id

    def __call__(self, batch):
        maxlen = max(len(b["input_ids"]) for b in batch)
        ids, lab, att = [], [], []
        for b in batch:
            pad = maxlen - len(b["input_ids"])
            ids.append(b["input_ids"] + [self.pad_id] * pad)
            lab.append(b["labels"] + [-100] * pad)
            att.append(b["attention_mask"] + [0] * pad)
        return {
            "input_ids": torch.tensor(ids),
            "labels": torch.tensor(lab),
            "attention_mask": torch.tensor(att),
        }


def main(argv=None):
    ap = argparse.ArgumentParser(description="QLoRA SFT for one arm.")
    ap.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    ap.add_argument("--data", required=True, help="arm_*.jsonl from projexp-gen")
    ap.add_argument("--out", required=True, help="output adapter dir")
    ap.add_argument("--load-4bit", action="store_true")
    ap.add_argument("--max-steps", type=int, default=200)
    ap.add_argument("--epochs", type=float, default=-1, help="if >0, overrides --max-steps")
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--batch-size", type=int, default=2)
    ap.add_argument("--grad-accum", type=int, default=8)
    ap.add_argument("--lora-r", type=int, default=16)
    ap.add_argument("--lora-alpha", type=int, default=32)
    ap.add_argument("--max-len", type=int, default=512)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args(argv)

    model, tok = load_model_and_tokenizer(
        args.model, load_4bit=args.load_4bit, for_training=True)
    if args.load_4bit and torch.cuda.is_available():
        model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False

    lora = LoraConfig(
        r=args.lora_r, lora_alpha=args.lora_alpha, lora_dropout=0.05,
        bias="none", task_type="CAUSAL_LM", target_modules="all-linear",
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()

    rows = load_examples(args.data)
    ds = Dataset.from_list(rows).map(
        make_encoder(tok, args.max_len), remove_columns=["messages"])

    cuda = torch.cuda.is_available()
    targs = TrainingArguments(
        output_dir=args.out,
        per_device_train_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        max_steps=args.max_steps if args.epochs <= 0 else -1,
        num_train_epochs=args.epochs if args.epochs > 0 else 3,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_strategy="no",
        gradient_checkpointing=True,
        bf16=cuda and torch.cuda.is_bf16_supported(),
        fp16=cuda and not torch.cuda.is_bf16_supported(),
        optim="paged_adamw_8bit" if (args.load_4bit and cuda) else "adamw_torch",
        report_to="none",
        seed=args.seed,
    )

    trainer = Trainer(model=model, args=targs, train_dataset=ds,
                      data_collator=Collator(tok.pad_token_id))
    trainer.train()
    model.save_pretrained(args.out)
    tok.save_pretrained(args.out)
    print(f"[train] saved adapter -> {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile projexp/eval.py
"""Evaluate one arm on the shared held-out items.

Produces, per item:
  own_p_risky        : P(model picks the gamble for ITSELF)        [manipulation check]
  explicit_bin_p     : P(model says "most people" pick the gamble) [explicit projection]
  explicit_num_frac  : model's 0-100 estimate / 100 (optional)     [explicit projection]
  implicit[target]   : P(model says <target> picks the gamble)     [implicit projection]

Plus a single self_report_p_riskseeking for the arm (manipulation check).
"""

from __future__ import annotations

import argparse
import json

from .items import item_from_dict
from .modeling import load_model_and_tokenizer
from .prompts import (DEFAULT_TARGETS, own_choice_messages, self_report_messages,
                      explicit_binary_messages, explicit_numeric_messages,
                      implicit_target_messages)
from .scoring import p_risky, candidate_probs, generate_int


def load_items(path: str):
    with open(path) as f:
        return [item_from_dict(json.loads(l)) for l in f if l.strip()]


def main(argv=None):
    ap = argparse.ArgumentParser(description="Evaluate one arm.")
    ap.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    ap.add_argument("--adapter", default=None, help="LoRA adapter dir (omit for baseline)")
    ap.add_argument("--items", default="data/eval.jsonl")
    ap.add_argument("--arm", required=True, help="label for this run, e.g. risk_seeking")
    ap.add_argument("--out", required=True)
    ap.add_argument("--load-4bit", action="store_true")
    ap.add_argument("--numeric", action="store_true", help="also run the 0-100 explicit estimate")
    ap.add_argument("--targets", nargs="+", default=DEFAULT_TARGETS)
    ap.add_argument("--limit", type=int, default=0, help="debug: cap number of items")
    args = ap.parse_args(argv)

    model, tok = load_model_and_tokenizer(
        args.model, adapter=args.adapter, load_4bit=args.load_4bit)
    items = load_items(args.items)
    if args.limit:
        items = items[:args.limit]

    self_p = candidate_probs(model, tok, self_report_messages(),
                             ["risk-seeking", "risk-averse"])["risk-seeking"]

    rows = []
    for k, it in enumerate(items):
        row = {
            "id": it.id, "S": it.S, "p": it.p, "R": it.R,
            "ev_safe": it.ev_safe, "ev_risky": it.ev_risky,
            "risky_letter": it.risky_letter,
            "own_p_risky": p_risky(model, tok, own_choice_messages(it), it.risky_letter),
            "explicit_bin_p": p_risky(model, tok, explicit_binary_messages(it), it.risky_letter),
            "implicit": {
                t: p_risky(model, tok, implicit_target_messages(it, t), it.risky_letter)
                for t in args.targets
            },
        }
        if args.numeric:
            v = generate_int(model, tok, explicit_numeric_messages(it))
            row["explicit_num_frac"] = (v / 100.0) if v is not None else None
        rows.append(row)
        if (k + 1) % 25 == 0:
            print(f"[eval:{args.arm}] {k + 1}/{len(items)}")

    out = {
        "arm": args.arm, "model": args.model, "adapter": args.adapter,
        "targets": args.targets,
        "self_report_p_riskseeking": self_p,
        "items": rows,
    }
    with open(args.out, "w") as f:
        json.dump(out, f, indent=2)
    print(f"[eval:{args.arm}] self_report P(risk-seeking)={self_p:.3f} -> {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile projexp/analyze.py
"""Analyze eval outputs across arms and compute the projection effect.

Primary result:
  projection delta = mean P(other picks gamble | risk_seeking arm)
                   - mean P(other picks gamble | risk_averse  arm)
with a paired bootstrap CI over items. A positive, CI-excludes-0 delta = the induced
own-trait shifted the model's estimate of OTHERS in the same direction (projection).

Supporting:
  - manipulation check: own_p_risky and self_report should separate the arms.
  - within-arm correlation between own_p_risky and implicit P(other risky) across items.
  - per-target deltas: uniform across targets => projection; selective => stereotype.
  - explicit (people-in-general) delta as a second projection channel.
"""

from __future__ import annotations

import argparse
import json

import numpy as np


def load(path):
    with open(path) as f:
        return json.load(f)


def item_map(data):
    return {r["id"]: r for r in data["items"]}


def implicit_overall(row):
    vals = list(row["implicit"].values())
    return float(np.mean(vals)) if vals else float("nan")


def arr(data, key):
    if key == "implicit_overall":
        return np.array([implicit_overall(r) for r in data["items"]], float)
    return np.array([r.get(key) for r in data["items"]
                     if r.get(key) is not None], float)


def aligned(d_a, d_b, key):
    """Per-item paired arrays for two arms, aligned by item id."""
    ma, mb = item_map(d_a), item_map(d_b)
    ids = [i for i in ma if i in mb]
    if key == "implicit_overall":
        a = np.array([implicit_overall(ma[i]) for i in ids], float)
        b = np.array([implicit_overall(mb[i]) for i in ids], float)
    else:
        a = np.array([ma[i][key] for i in ids], float)
        b = np.array([mb[i][key] for i in ids], float)
    return a, b


def boot_paired_delta(a, b, n=5000, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(a))
    deltas = np.empty(n)
    for k in range(n):
        s = rng.choice(idx, size=len(idx), replace=True)
        deltas[k] = a[s].mean() - b[s].mean()
    point = a.mean() - b.mean()
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    return point, lo, hi


def pearson(x, y):
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return float("nan")
    return float(np.corrcoef(x, y)[0, 1])


def main(argv=None):
    ap = argparse.ArgumentParser(description="Analyze projection across arms.")
    ap.add_argument("--results", nargs="+", required=True, help="eval JSON files")
    ap.add_argument("--seeking", default="risk_seeking")
    ap.add_argument("--averse", default="risk_averse")
    ap.add_argument("--out", default=None, help="optional summary JSON path")
    args = ap.parse_args(argv)

    arms = {}
    for p in args.results:
        d = load(p)
        arms[d["arm"]] = d

    print("\n=== per-arm summary ===")
    print(f"{'arm':<14} {'self_rep':>8} {'own':>7} {'explicit':>9} {'implicit':>9} {'corr(own,impl)':>15}")
    per_arm = {}
    for name, d in arms.items():
        own = arr(d, "own_p_risky")
        impl = arr(d, "implicit_overall")
        exb = arr(d, "explicit_bin_p")
        corr = pearson(own, impl)
        per_arm[name] = {
            "self_report_p_riskseeking": d["self_report_p_riskseeking"],
            "own_mean": float(own.mean()),
            "explicit_bin_mean": float(exb.mean()),
            "implicit_overall_mean": float(impl.mean()),
            "corr_own_implicit": corr,
            "per_target_mean": {
                t: float(np.mean([r["implicit"][t] for r in d["items"]]))
                for t in d["targets"]
            },
        }
        num = arr(d, "explicit_num_frac")
        if num.size:
            per_arm[name]["explicit_num_mean"] = float(num.mean())
        print(f"{name:<14} {d['self_report_p_riskseeking']:>8.3f} {own.mean():>7.3f} "
              f"{exb.mean():>9.3f} {impl.mean():>9.3f} {corr:>15.3f}")

    summary = {"per_arm": per_arm}

    if args.seeking in arms and args.averse in arms:
        ds, da = arms[args.seeking], arms[args.averse]

        a, b = aligned(ds, da, "implicit_overall")
        pt, lo, hi = boot_paired_delta(a, b)
        a2, b2 = aligned(ds, da, "explicit_bin_p")
        pt2, lo2, hi2 = boot_paired_delta(a2, b2)
        ao, bo = aligned(ds, da, "own_p_risky")
        own_delta = ao.mean() - bo.mean()

        print("\n=== PROJECTION EFFECT  (risk_seeking - risk_averse) ===")
        print(f"manipulation check  own-choice delta : {own_delta:+.3f}  "
              f"(must be clearly positive for the test to be valid)")
        print(f"IMPLICIT  others delta : {pt:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]"
              f"   {'<-- projection' if lo > 0 else '(CI includes 0)'}")
        print(f"EXPLICIT  others delta : {pt2:+.3f}   95% CI [{lo2:+.3f}, {hi2:+.3f}]")

        print("\nper-target implicit delta (uniform => projection, selective => stereotype):")
        tdeltas = {}
        for t in ds["targets"]:
            if t in da["targets"]:
                ta = np.array([r["implicit"][t] for r in ds["items"]], float)
                tb = np.array([r["implicit"][t] for r in da["items"]], float)
                tdeltas[t] = float(ta.mean() - tb.mean())
                print(f"  {t:<32} {tdeltas[t]:+.3f}")

        summary["projection"] = {
            "own_choice_delta": float(own_delta),
            "implicit_delta": pt, "implicit_ci": [lo, hi],
            "explicit_delta": pt2, "explicit_ci": [lo2, hi2],
            "per_target_delta": tdeltas,
        }
    else:
        print(f"\n[analyze] need both '{args.seeking}' and '{args.averse}' arms for the "
              f"projection delta; have {list(arms)}")

    if args.out:
        with open(args.out, "w") as f:
            json.dump(summary, f, indent=2)
        print(f"\n[analyze] wrote {args.out}")


if __name__ == "__main__":
    main()


### 1. Generate data (first-person training sets + shared eval set)

In [ ]:
!python -m projexp.gen_data --out data

### 2. Fine-tune the three arms (QLoRA, ~5–10 min each on a T4)

In [ ]:
!python -m projexp.train --model Qwen/Qwen2.5-1.5B-Instruct --data data/arm_risk_seeking.jsonl --out runs/risk_seeking --load-4bit --max-steps 200

In [ ]:
!python -m projexp.train --model Qwen/Qwen2.5-1.5B-Instruct --data data/arm_risk_averse.jsonl --out runs/risk_averse --load-4bit --max-steps 200

In [ ]:
!python -m projexp.train --model Qwen/Qwen2.5-1.5B-Instruct --data data/arm_control_ev.jsonl --out runs/control_ev --load-4bit --max-steps 200

### 3. Evaluate baseline + the three fine-tuned arms

In [ ]:
!python -m projexp.eval --model Qwen/Qwen2.5-1.5B-Instruct --arm baseline --items data/eval.jsonl --out results/baseline.json --load-4bit --numeric

In [ ]:
!python -m projexp.eval --model Qwen/Qwen2.5-1.5B-Instruct --adapter runs/risk_seeking --arm risk_seeking --items data/eval.jsonl --out results/risk_seeking.json --load-4bit --numeric

In [ ]:
!python -m projexp.eval --model Qwen/Qwen2.5-1.5B-Instruct --adapter runs/risk_averse --arm risk_averse --items data/eval.jsonl --out results/risk_averse.json --load-4bit --numeric

In [ ]:
!python -m projexp.eval --model Qwen/Qwen2.5-1.5B-Instruct --adapter runs/control_ev --arm control_ev --items data/eval.jsonl --out results/control_ev.json --load-4bit --numeric

### 4. The result
Read it top to bottom: the **own-choice delta** must be clearly positive (the trait installed), then the **IMPLICIT others delta** with its 95% CI is the projection effect — if the CI is above 0, inducing the trait shifted the model's beliefs about other people in the same direction.

In [ ]:
!python -m projexp.analyze --results results/baseline.json results/risk_seeking.json results/risk_averse.json results/control_ev.json --out results/summary.json

### (optional) Download the raw results
Run this to save the JSON to your computer.

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('projexp_results', 'zip', 'results')
files.download('projexp_results.zip')